# CamemBERT Training Interface

This notebook is a thin orchestration layer around `BERT/train.py`. It does not duplicate the training loop, model definition, checkpointing, scheduler, early stopping, or metrics logic. Edit the configuration cells, then run the training cell.

## 1. Import the Existing Training Pipeline

The notebook adds the `BERT/` directory to `sys.path`, imports the existing config dataclasses, and imports `run_training()` from `train.py`.

In [ ]:
from argparse import Namespace
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT / "BERT").exists():
    BERT_DIR = PROJECT_ROOT / "BERT"
else:
    BERT_DIR = PROJECT_ROOT
    PROJECT_ROOT = BERT_DIR.parent

if str(BERT_DIR) not in sys.path:
    sys.path.insert(0, str(BERT_DIR))

from config import ModelConfig, PathConfig, TrainingConfig
import train as train_pipeline

paths = PathConfig()
model_defaults = ModelConfig()
train_defaults = TrainingConfig()

print(f"Project root: {PROJECT_ROOT}")
print(f"BERT module directory: {BERT_DIR}")

## 2. Configure the Training Run

Change these values as needed. To resume from a checkpoint, set `MODEL_NAME` to a saved checkpoint directory such as `BERT/outputs/checkpoints/epoch_03`.

In [ ]:
DATA_DIR = paths.data_dir
MODEL_NAME = model_defaults.model_name  # Example resume value: BERT_DIR / "outputs" / "checkpoints" / "epoch_03"

MAX_LENGTH = model_defaults.max_length
BATCH_SIZE = train_defaults.batch_size
EVAL_BATCH_SIZE = train_defaults.eval_batch_size
EPOCHS = train_defaults.epochs
LEARNING_RATE = train_defaults.learning_rate
WEIGHT_DECAY = train_defaults.weight_decay
WARMUP_RATIO = train_defaults.warmup_ratio
MAX_GRAD_NORM = train_defaults.max_grad_norm
PATIENCE = train_defaults.patience
NUM_WORKERS = train_defaults.num_workers
SEED = train_defaults.seed
NO_AMP = False

CHECKPOINT_DIR = paths.checkpoint_dir
BEST_MODEL_DIR = paths.best_model_dir
LOG_DIR = paths.log_dir
ARTIFACT_DIR = paths.artifact_dir

print("Training configuration ready.")
print(f"Data: {DATA_DIR}")
print(f"Model source: {MODEL_NAME}")
print(f"Best model output: {BEST_MODEL_DIR}")

## 3. Launch Training

This cell builds the same argument namespace that `python BERT/train.py ...` would create, then calls `train_pipeline.run_training(args)` from `train.py`.

In [ ]:
args = Namespace(
    data_dir=Path(DATA_DIR),
    model_name=str(MODEL_NAME),
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=MAX_GRAD_NORM,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    no_amp=NO_AMP,
    checkpoint_dir=Path(CHECKPOINT_DIR),
    best_model_dir=Path(BEST_MODEL_DIR),
    log_dir=Path(LOG_DIR),
    artifact_dir=Path(ARTIFACT_DIR),
)

metrics = train_pipeline.run_training(args)
metrics["paths"]

## 4. Inspect Training History

After training completes, this cell loads the history CSV generated by `train.py`.

In [ ]:
import pandas as pd

history_path = Path(LOG_DIR) / "history.csv"
if history_path.exists():
    history_df = pd.read_csv(history_path)
    display(history_df)
else:
    print(f"History file not found yet: {history_path}")

## 5. Output Locations

The training pipeline writes:

- Best model: `BERT/outputs/models/best_model/`
- Per-epoch checkpoints: `BERT/outputs/checkpoints/epoch_XX/`
- Training history: `BERT/outputs/logs/history.csv`
- Metrics: `BERT/artifacts/metrics.json`

Use `BERT/evaluate.py` after training to evaluate the saved best model on the test split.